In [0]:
%sql
-- Databricks notebook source
-- MAGIC %md
-- MAGIC # Validations
-- MAGIC Checks after gold layer (also used to verify Genie).

-- COMMAND ----------

-- 1) Fact rows should match silver (1:1 on result_id)
SELECT
  (SELECT COUNT(*) FROM marathos.silver.races_obt) AS silver_cnt,
  (SELECT COUNT(*) FROM marathos.gold.fct_results) AS fact_cnt;

-- COMMAND ----------

-- 2) No orphan event_ids
SELECT COUNT(*) AS orphan_events
FROM marathos.gold.fct_results f
LEFT JOIN marathos.gold.dim_event e ON f.event_id = e.event_id
WHERE e.event_id IS NULL;

-- COMMAND ----------

-- 3) Speed within expected marathon range
SELECT MIN(speed_kmh), MAX(speed_kmh), AVG(speed_kmh)
FROM marathos.gold.fct_results;

-- COMMAND ----------

-- 4) Avg finish time (hours), USA, 2018
SELECT
  AVG(performance_seconds) / 3600.0 AS avg_hours
FROM marathos.gold.fct_results f
JOIN marathos.gold.dim_athlete a ON f.athlete_id = a.athlete_id
JOIN marathos.gold.dim_event e ON f.event_id = e.event_id
WHERE a.athlete_country = 'USA' AND e.event_year = 2018;
